# Example 1: Basic INT8 Quantization

This notebook demonstrates the fundamentals of INT8 quantization including:
- Symmetric quantization
- Asymmetric quantization
- Quantization error analysis
- GPU-accelerated quantization

**Learning Objectives:**
- Understand how quantization reduces memory
- Learn symmetric vs asymmetric quantization
- Implement per-tensor and per-channel quantization
- Analyze quantization error metrics

## Setup and Imports

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import time
from typing import Tuple

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")

In [ ]:
# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## Part 1: Symmetric Quantization

Symmetric quantization maps values symmetrically around zero:

$$Q(x) = \text{round}\left(\frac{x}{S}\right)$$

where $S = \frac{\max(|x|)}{2^{bits-1} - 1}$

This is ideal for data with symmetric distributions (e.g., weights centered around zero).

In [ ]:
def symmetric_quantize(tensor: torch.Tensor, bits: int = 8) -> Tuple[torch.Tensor, float]:
    """
    Symmetric quantization: maps values symmetrically around zero.
    
    Args:
        tensor: Input tensor to quantize
        bits: Number of bits for quantization (default: 8)
    
    Returns:
        Quantized tensor and scale factor
    """
    qmax = 2 ** (bits - 1) - 1
    qmin = -2 ** (bits - 1)
    
    max_val = torch.max(torch.abs(tensor))
    scale = max_val / qmax
    
    quantized = torch.clamp(torch.round(tensor / scale), qmin, qmax)
    
    return quantized.to(torch.int8), scale.item()

def symmetric_dequantize(quantized: torch.Tensor, scale: float) -> torch.Tensor:
    """Dequantize a symmetrically quantized tensor."""
    return quantized.float() * scale

### Test Symmetric Quantization

In [ ]:
# Create a sample tensor
symmetric_tensor = torch.randn(1000, 1000, device=device)

print(f"Original tensor shape: {symmetric_tensor.shape}")
print(f"Original tensor range: [{symmetric_tensor.min():.4f}, {symmetric_tensor.max():.4f}]")
print(f"Original memory: {symmetric_tensor.element_size() * symmetric_tensor.nelement() / 1e6:.2f} MB")

# Quantize
q_sym, scale_sym = symmetric_quantize(symmetric_tensor)
print(f"\nQuantized memory: {q_sym.element_size() * q_sym.nelement() / 1e6:.2f} MB")
print(f"Memory reduction: {symmetric_tensor.element_size() / q_sym.element_size():.1f}x")
print(f"Scale factor: {scale_sym:.6f}")

# Dequantize
dq_sym = symmetric_dequantize(q_sym, scale_sym)

## Part 2: Asymmetric Quantization

Asymmetric quantization maps values with separate min/max:

$$Q(x) = \text{round}\left(\frac{x}{S}\right) - Z$$

where:
- $S = \frac{\max - \min}{2^{bits} - 1}$
- $Z = -\text{round}\left(\frac{\min}{S}\right)$

This is better for asymmetric distributions (e.g., activations after ReLU).

In [ ]:
def asymmetric_quantize(tensor: torch.Tensor, bits: int = 8) -> Tuple[torch.Tensor, float, int]:
    """
    Asymmetric quantization: maps values with separate min/max.
    
    Args:
        tensor: Input tensor to quantize
        bits: Number of bits for quantization
    
    Returns:
        Quantized tensor, scale factor, and zero point
    """
    qmax = 2 ** bits - 1
    qmin = 0
    
    min_val = torch.min(tensor)
    max_val = torch.max(tensor)
    
    scale = (max_val - min_val) / (qmax - qmin)
    zero_point = qmin - torch.round(min_val / scale)
    
    quantized = torch.clamp(torch.round(tensor / scale) + zero_point, qmin, qmax)
    
    return quantized.to(torch.uint8), scale.item(), int(zero_point.item())

def asymmetric_dequantize(quantized: torch.Tensor, scale: float, zero_point: int) -> torch.Tensor:
    """Dequantize an asymmetrically quantized tensor."""
    return (quantized.float() - zero_point) * scale

### Compare Symmetric vs Asymmetric on Asymmetric Data

In [ ]:
# Create asymmetric data (like ReLU output)
asymmetric_tensor = torch.relu(torch.randn(1000, 1000, device=device))

print(f"Asymmetric tensor range: [{asymmetric_tensor.min():.4f}, {asymmetric_tensor.max():.4f}]")

# Symmetric quantization
q_sym2, scale_sym2 = symmetric_quantize(asymmetric_tensor)
dq_sym2 = symmetric_dequantize(q_sym2, scale_sym2)

# Asymmetric quantization
q_asym, scale_asym, zp_asym = asymmetric_quantize(asymmetric_tensor)
dq_asym = asymmetric_dequantize(q_asym, scale_asym, zp_asym)

# Calculate errors
def calculate_error(original, reconstructed):
    mse = torch.mean((original - reconstructed) ** 2).item()
    mae = torch.mean(torch.abs(original - reconstructed)).item()
    signal_power = torch.mean(original ** 2).item()
    sqnr_db = 10 * np.log10(signal_power / mse) if mse > 0 else float('inf')
    return {'mse': mse, 'mae': mae, 'sqnr_db': sqnr_db}

metrics_sym2 = calculate_error(asymmetric_tensor, dq_sym2)
metrics_asym = calculate_error(asymmetric_tensor, dq_asym)

print(f"\n📊 Symmetric quantization:")
print(f"   SQNR: {metrics_sym2['sqnr_db']:.2f} dB")
print(f"   MAE: {metrics_sym2['mae']:.6f}")

print(f"\n📊 Asymmetric quantization:")
print(f"   SQNR: {metrics_asym['sqnr_db']:.2f} dB")
print(f"   MAE: {metrics_asym['mae']:.6f}")

improvement = metrics_asym['sqnr_db'] - metrics_sym2['sqnr_db']
print(f"\n✓ Asymmetric improved SQNR by {improvement:.2f} dB for asymmetric data!")

## Part 3: Neural Network Weight Quantization

Let's apply quantization to a simple neural network.

In [ ]:
class SimpleNN(nn.Module):
    """A simple neural network for demonstration."""
    def __init__(self, input_size=784, hidden_size=512, output_size=10):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.fc3 = nn.Linear(hidden_size, output_size)
    
    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        return x

# Create model
model = SimpleNN().to(device)
num_params = sum(p.numel() for p in model.parameters())
original_size = sum(p.element_size() * p.nelement() for p in model.parameters()) / 1e6

print(f"Model parameters: {num_params:,}")
print(f"Original model size: {original_size:.2f} MB")

In [ ]:
# Quantize all weights
def quantize_model_weights(model: nn.Module) -> dict:
    quantized_state = {}
    
    for name, param in model.named_parameters():
        if 'weight' in name:
            q_weight, scale = symmetric_quantize(param.data)
            quantized_state[name] = {'quantized': q_weight, 'scale': scale}
        else:
            quantized_state[name] = {'float': param.data}
    
    return quantized_state

quantized_state = quantize_model_weights(model)

# Calculate size
quantized_size = sum(
    v['quantized'].element_size() * v['quantized'].nelement()
    for v in quantized_state.values()
    if 'quantized' in v
) / 1e6

print(f"Quantized model size: {quantized_size:.2f} MB")
print(f"Size reduction: {original_size / quantized_size:.2f}x")

## Part 4: Per-Channel vs Per-Tensor Quantization

**Per-tensor:** One scale factor for entire tensor  
**Per-channel:** One scale factor per output channel (better accuracy)

In [ ]:
# Get first layer weights
weight = model.fc1.weight.data
print(f"Layer weight shape: {weight.shape}")

# Per-tensor quantization
q_tensor, scale_tensor = symmetric_quantize(weight)
dq_tensor = symmetric_dequantize(q_tensor, scale_tensor)
metrics_tensor = calculate_error(weight, dq_tensor)

print(f"\n📊 Per-tensor quantization:")
print(f"   Single scale factor: {scale_tensor:.6f}")
print(f"   SQNR: {metrics_tensor['sqnr_db']:.2f} dB")

# Per-channel quantization
per_channel_quantized = []
scales = []

for i in range(weight.shape[0]):
    channel = weight[i:i+1, :]
    q_channel, scale = symmetric_quantize(channel)
    per_channel_quantized.append(q_channel)
    scales.append(scale)

# Dequantize per-channel
dq_channels = [symmetric_dequantize(q, s) for q, s in zip(per_channel_quantized, scales)]
dq_per_channel = torch.cat(dq_channels, dim=0)
metrics_channel = calculate_error(weight, dq_per_channel)

print(f"\n📊 Per-channel quantization:")
print(f"   {len(scales)} scale factors (one per output channel)")
print(f"   Scale range: [{min(scales):.6f}, {max(scales):.6f}]")
print(f"   SQNR: {metrics_channel['sqnr_db']:.2f} dB")

improvement = metrics_channel['sqnr_db'] - metrics_tensor['sqnr_db']
print(f"\n✓ Per-channel improved SQNR by {improvement:.2f} dB!")

## Summary

### Key Takeaways:
1. ✅ INT8 quantization reduces memory by **4×** (FP32 → INT8)
2. ✅ Asymmetric quantization is better for non-symmetric distributions
3. ✅ Per-channel quantization preserves more information
4. ✅ Quantization introduces small errors (trade-off for efficiency)
5. ✅ GPU acceleration works for both quantized and non-quantized ops

### Next Steps:
- Run notebook 02 for 4-bit quantization
- Run notebook 03 for real LLM quantization with bitsandbytes